In [3]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import os
warnings.simplefilter('ignore') # filter some warning messages
xr.set_options(display_style="html")  #display dataset nicely 

In [4]:
%%time

import coiled

cluster_type = "Coiled"

if cluster_type == "Coiled":
    cluster = coiled.Cluster(
        region="eu-central-1",
        arm=True,  # run on ARM to save energy & cost
        worker_vm_types=["t4g.medium"],  # cheap ARM instances
        worker_options={"nthreads": 2},
        n_workers=200,
        name='Oriana',
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        software="protocoast-notebook-arm",
        workspace="esip-lab",
        timeout=180  # keep cluster alive for 3 min
    )

Output()

CPU times: user 2.46 s, sys: 214 ms, total: 2.67 s
Wall time: 41.3 s


ClusterCreationError: Cluster status is error (reason: Workers all had error -> Instance Stopped: 
Coiled was unable to provision this EC2 Instance because of your current AWS account quotas.

We were unable to provision any VMs for this cluster with your existing quotas and current usage.

We recommend you use the Coiled CLI to submit a request to AWS to increase quotas:
    coiled setup aws --quotas --region eu-central-1


Full error from AWS should indicate the relevant quotas:
VcpuLimitExceeded - You have requested more vCPU capacity than your current vCPU limit of 384 allows for the instance bucket that the specified instance type belongs to. Please visit http://aws.amazon.com/contact-us/ec2-request to request an adjustment to this limit.
UnfulfillableCapacity - Failed to fulfill capacity. Please review errors in the response.) (cluster_id: 1579841)

In [ ]:
client = cluster.get_client()
client

In [ ]:
%%time

ds_sst = xr.open_zarr('https://mur-sst.s3.us-west-2.amazonaws.com/zarr-v1',consolidated=True)

ds_sst

In [ ]:
sst = ds_sst['analysed_sst']
sst_subset = sst.sel(
    lat=slice(-25, 25),   # reversed if lat is descending
    lon=slice(-180, -70)
)

In [ ]:
# MONTHLY MEAN 
sst_monthly = sst_subset.resample(time='1MS').mean('time', keep_attrs=True, skipna=False)

# MONTHLY CLIMATOLOGY ---
climatology_mean_monthly = sst_monthly.groupby('time.month').mean('time', keep_attrs=True, skipna=False)

# MONTHLY ANOMALY ---
sst_anomaly_monthly = (sst_monthly.groupby('time.month') - climatology_mean_monthly)

# OUTPUT 
sst_anomaly_monthly

In [ ]:
d_spatial = (
    sst_anomaly_monthly
    .sel(time=slice("2015-12-01", "2016-01-31"))
    .mean(dim="time")
)

In [ ]:
d_spatial = data_2015.mean(dim="time", skipna=True)

In [ ]:
# --- Plot spatial map (lat-lon) ---
d_spatial.plot(cmap="RdBu_r")

In [ ]:
d = sst_anomaly_monthly.sel(time=slice('2015', '2022')).mean(dim=["lat", "lon"]).compute()
d.plot()

In [ ]:
import hvplot.xarray
import numpy as np
import holoviews as hv

In [ ]:
#plot#
d_spatial.hvplot.quadmesh(
    x='lon',
    y='lat',
    rasterize=True,
    frame_width=700,
    cmap='RdBu_r',
    geo=True,
    tiles='OSM',
    clim=(-2, 2),
    title='SST Anomaly (2015 Mean)'
)

In [ ]:
#plot#
d_spatial.hvplot.quadmesh(
    x='lon',
    y='lat',
    rasterize=True,
    frame_width=700,
    cmap='RdBu_r',
    geo=True,
    tiles='OSM',
    clim=(-2, 2),
    title='SST Anomaly (2015 Mean)'
)